In [ ]:
!pip install torch_snippets # instala biblioteca para PyTorch
from torch_snippets import * # importa as funções da biblioteca snippets
from torchvision import transforms as T # importa ferramentas de transformação de imagem
from torch.nn import functional as F # importa funções
from torchvision.models import vgg19 # importa a arquitetura da rede
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
preprocess = T.Compose([ # define sequência
    T.ToTensor(), # converte imagem para tensor
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), # normalização padrão
    T.Lambda(lambda x: x.mul_(255)) # escala valores
])

postprocess = T.Compose([ # define sequência
    T.Lambda(lambda x: x.mul_(1./255)), # volta valores
    T.Normalize(mean=[-0.485/0.229, -0.456/0.224,-0.406/0.225], std=[1/0.229,1/0.224,1/0.255]), # reverte a normalização
])

In [ ]:
class GramMatrix(nn.Module): # define a matriz
    def forward(self,input): # processo de cálculo
        b,c,h,w = input.size() # obtém dimensões
        feat = input.view(b,c,h*w)
        G = feat@feat.transpose(1,2) # multiplica a matriz
        G.div_(h*w) # normaliza dividindo
        return G # retorna a matriz

class GramMSELoss(nn.Module): # define perda de estilo
    def forward(self,input,target): # cálculo do erro
        out = F.mse_loss(GramMatrix()(input),target)
        return(out) # retorna o erro

class vgg19_modified(nn.Module):
    def __init__(self): # inicializa o modelo
        super().__init__() # herda da classe mãe
        features = list(vgg19(pretrained=True).features) # carrega pesos
        self.features = nn.ModuleList(features).eval() # congela camadas

    def forward(self, x, layers=[]): # fluxo com extração de camadas
        order = np.argsort(layers) # ordena os índices das camadas
        _results, results = [], [] # listas para armazenar

        for ix, model in enumerate(self.features): # percorre todas as camadas
            x = model(x) # passa a imagem
            if ix in layers: # se a camada for um alvo
                _results.append(x) # armazena o mapa de características

        for o in order: # organiza os resultados
            results.append(_results[o])

        return results if layers is not [] else x # retorna lista

In [ ]:
vgg = vgg19_modified().to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:05<00:00, 112MB/s]


In [ ]:
!wget https://easydrawingguides.com/wp-content/uploads/2016/10/how-to-draw-an-elephant-featured-image-1200.png
!wget https://www.neh.gov/sites/default/files/2022-09/Fall_2022_web-images_Picasso_32.jpg

--2026-04-23 01:20:18--  https://easydrawingguides.com/wp-content/uploads/2016/10/how-to-draw-an-elephant-featured-image-1200.png
Resolving easydrawingguides.com (easydrawingguides.com)... 104.19.154.92
Connecting to easydrawingguides.com (easydrawingguides.com)|104.19.154.92|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 56936 (56K) [image/png]
Saving to: ‘how-to-draw-an-elephant-featured-image-1200.png’

how-to-draw-an-elep 100%[===================>]  55.60K  --.-KB/s    in 0.002s  

2026-04-23 01:20:18 (35.3 MB/s) - ‘how-to-draw-an-elephant-featured-image-1200.png’ saved [56936/56936]

--2026-04-23 01:20:18--  https://www.neh.gov/sites/default/files/2022-09/Fall_2022_web-images_Picasso_32.jpg
Resolving www.neh.gov (www.neh.gov)... 23.21.228.79
Connecting to www.neh.gov (www.neh.gov)|23.21.228.79|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5309491 (5.1M) [image/jpeg]
Saving to: ‘Fall_2022_web-images_Picasso_32.jpg’

Fall_2022_w

In [ ]:
!ls

Fall_2022_web-images_Picasso_32.jpg		 sample_data
how-to-draw-an-elephant-featured-image-1200.png


In [ ]:
imgs = [Image.open(path).resize((512,512)).convert('RGB') for path in ['Fall_2022_web-images_Picasso_32.jpg','how-to-draw-an-elephant-featured-image-1200.png']]
style_image, content_image = [preprocess(img).to(device)[None] for img in imgs]

In [ ]:
opt_img = content_image.data.clone()
opt_img.requires_grad = True

In [ ]:
style_layers = [0, 5, 10, 19, 28] # define índices das camadas
content_layers = [21] # define índice da camada
loss_layers = style_layers + content_layers # combina todas as camadas

In [ ]:
loss_fns = [GramMSELoss()] * len(style_layers) + [nn.MSELoss()] * len(content_layers) # cria lista de funções
loss_fns = [loss_fn.to(device) for loss_fn in loss_fns] # move todas as funçõ

In [ ]:
style_weights = [1000/n**2 for n in [64,128,256,512,512]] # calcula pesos
content_weights = [1] # define o peso para a preservação
weights = style_weights + content_weights # combina todos os pesos

In [ ]:
style_target = [GramMatrix()(A).detach() for A in vgg(style_image, style_layers)] # extrai e fixa as matrizes
content_targets = [A.detach() for A in vgg(content_image, content_layers)] # extrai e fixa
targets = style_target + content_targets # consolida todos os alvos

In [ ]:
max_iters = 500 # define o limite máximo
optimizer = optim.LBFGS([opt_img]) # usa LBFGS para otimizar os pixels
log = Report(max_iters) # inicializa o sistema

In [ ]:
iters = 0 # contador
while iters < max_iters: # loop principal
    def closure(): # função necessária
        global iters # acessa a variável
        iters += 1 # incrementa o contador
        optimizer.zero_grad() # zera os gradientes
        out = vgg(opt_img, loss_layers) # extrai mapas de características
        layer_losses = [weights[a] * loss_fns[a](A, targets[a]) for a,A in enumerate(out)]
        loss = sum(layer_losses) # soma todas as perdas
        loss.backward() # calcula os gradientes
        log.record(pos=iters, loss=loss, end='\r') # registra o progresso
        return loss # retorna a perda
    optimizer.step(closure) # executa o passo de otimização

EPOCH: 89.000  loss: 2537166.250  (2492.28s - 11509.29s remaining)

In [ ]:
log.plot(log=True)

In [ ]:
with torch.no_grad():
    out_img = postprocess(opt_img[0]).permute(1,2,0)
show(out_img)